# Part 0: Data Prepare

In [ ]:
# !pip install wrds pandas numpy

In [45]:
import wrds
import pandas as pd
import numpy as np

## 1. Connect to WRDS

In [46]:
db = wrds.Connection()

WRDS recommends setting up a .pgpass file.
Created .pgpass file successfully.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done


## 2. Download the Compustat Annual Financial Data

The query below extends the original Compustat pull to include four variables required for Part II analysis: `dltt` (long-term debt), `dlc` (debt in current liabilities), `xrd` (research and development expense), and `ppent` (net property, plant and equipment). These are retained in the final panel and saved in the CSV so that Part II can construct leverage, R&D intensity, and capital intensity at the firm level without an additional WRDS connection.

In [47]:
# Original query (before Part II additions):
# comp = db.raw_sql("""
# SELECT
#     f.gvkey, f.datadate, f.fyear, f.fyr,
#     f.at, f.sale, f.ni, f.ceq, f.cusip, f.sich AS sic
# FROM comp.funda f
# WHERE
#     f.indfmt = 'INDL' AND f.datafmt = 'STD'
#     AND f.consol = 'C' AND f.popsrc = 'D'
#     AND f.fyear BETWEEN 2013 AND 2023
# """)

# Extended for Part II: added dltt, dlc, xrd, and ppent
comp = db.raw_sql("""
SELECT
    f.gvkey,
    f.datadate,
    f.fyear,
    f.fyr,
    f.at,
    f.sale,
    f.ni,
    f.ceq,
    f.cusip,
    f.sich AS sic,
    f.dltt,
    f.dlc,
    f.xrd,
    f.ppent
FROM comp.funda f
WHERE
    f.indfmt = 'INDL'
    AND f.datafmt = 'STD'
    AND f.consol = 'C'
    AND f.popsrc = 'D'
    AND f.fyear BETWEEN 2013 AND 2023
""")

In [48]:
comp['datadate'] = pd.to_datetime(comp['datadate'])
comp['cusip8'] = comp['cusip'].astype(str).str[:8]

In [49]:
# keep total assets > 0
comp = comp[comp['at'] > 0].copy()

In [50]:
# Construct basic variables
comp['roa']  = comp['ni'] / comp['at']
comp['size'] = np.log(comp['at'])
comp['year'] = comp['datadate'].dt.year

In [51]:
print(f"Compustat rows: {comp.shape[0]}")

Compustat rows: 91829


## 3. Download the Monthly CRSP Data

In [52]:
crsp = db.raw_sql("""
SELECT
    m.permno,
    m.date,
    m.ret,
    m.prc,
    m.shrout,
    e.shrcd,
    e.exchcd
FROM crsp.msf m
INNER JOIN crsp.msenames e
    ON m.permno = e.permno
    AND m.date BETWEEN e.namedt AND COALESCE(e.nameendt, '2099-12-31')
WHERE
    m.date BETWEEN '2013-01-01' AND '2025-12-31'
    AND e.shrcd IN (10, 11)
    AND e.exchcd IN (1, 2, 3)
""")

In [53]:
crsp['date']   = pd.to_datetime(crsp['date'])
crsp['ret']    = pd.to_numeric(crsp['ret'],    errors='coerce')
crsp['prc']    = pd.to_numeric(crsp['prc'],    errors='coerce')
crsp['shrout'] = pd.to_numeric(crsp['shrout'], errors='coerce')
crsp['me']     = crsp['prc'].abs() * crsp['shrout']
crsp['year']   = crsp['date'].dt.year
crsp['month']  = crsp['date'].dt.month

In [54]:
print(f"CRSP (after filtering): {crsp.shape[0]}")

CRSP (after filtering): 549323


In [55]:
# Retain valid monthly earnings
crsp_ret = crsp[['permno', 'date', 'ret']].dropna(subset=['ret']).copy()

In [56]:
def compute_post_fy_return(permno_series, datadate_series, crsp_ret_df):
    """
    For each (permno, datadate) record,
    The monthly earnings from the first to the twelfth month after taking the datadate
    Calculate the compound annualized return (1+r1)*(1+r2)*... *(1+r12) -1.
    If the number of months is less than 12, return NaN and also return the actual number of months.
    """
    records = []
    crsp_grouped = crsp_ret_df.groupby('permno')

    for permno, datadate in zip(permno_series, datadate_series):
        if permno not in crsp_grouped.groups:
            records.append({'future_annual_ret': np.nan, 'future_n_months_ret': 0})
            continue

        sub = crsp_grouped.get_group(permno).copy()
        # Take the monthly observations after datadate and sort them by date
        sub = sub[sub['date'] > datadate].sort_values('date')
        # The maximum is 12 months
        sub12 = sub.head(12)
        n = len(sub12)

        if n < 12:
            records.append({'future_annual_ret': np.nan, 'future_n_months_ret': n})
        else:
            compound_ret = (1 + sub12['ret']).prod() - 1
            records.append({'future_annual_ret': compound_ret, 'future_n_months_ret': n})

    return pd.DataFrame(records)

## 4. Download and Clean the CCM Link Table

In [57]:
ccm = db.raw_sql("""
SELECT
    gvkey,
    lpermno AS permno,
    linktype,
    linkprim,
    linkdt,
    linkenddt
FROM crsp.ccmxpf_linktable
WHERE
    linktype IN ('LC', 'LU', 'LS')
    AND linkprim IN ('C', 'P')
""")

In [58]:
ccm['linkdt']    = pd.to_datetime(ccm['linkdt'])
ccm['linkenddt'] = pd.to_datetime(ccm['linkenddt'])

## 5. Merge Compustat + CCM and Retain the Valid Link Window

In [59]:
comp_ccm = comp.merge(ccm, on='gvkey', how='left')

In [60]:
comp_ccm = comp_ccm[
    (comp_ccm['datadate'] >= comp_ccm['linkdt']) &
    (
        (comp_ccm['datadate'] <= comp_ccm['linkenddt']) |
        (comp_ccm['linkenddt'].isna())
    )
].copy()

In [61]:
print(f"Comp+CCM rows after merge: {comp_ccm.shape[0]}")

Comp+CCM rows after merge: 55445


## 6. Download ESG Data (item=1 = Overall ESG score)

In [62]:
dfs = []
for y in range(2013, 2024):
    print(f"download ESG score {y}...")
    tmp = db.raw_sql(f"""
    SELECT
        orgpermid,
        fy,
        value_ AS esg_score
    FROM tr_esg.esgscores
    WHERE item = 1        -- item=1: ESGScore（overall score，not Combined）
      AND fy = {y}
    """)
    dfs.append(tmp)

download ESG score 2013...
download ESG score 2014...
download ESG score 2015...
download ESG score 2016...
download ESG score 2017...
download ESG score 2018...
download ESG score 2019...
download ESG score 2020...
download ESG score 2021...
download ESG score 2022...
download ESG score 2023...


In [63]:
esg = pd.concat(dfs, ignore_index=True)
esg = esg.dropna(subset=['orgpermid', 'fy', 'esg_score']).copy()
esg = esg.groupby(['orgpermid', 'fy'], as_index=False)['esg_score'].mean()

In [64]:
print(f"ESG rows: {esg.shape[0]}")

ESG rows: 91770


## 7. Download the ESG Reference Table (obtain CUSIP for integration with Compustat)

In [65]:
org_list = esg['orgpermid'].dropna().astype('int64').unique().tolist()

In [66]:
chunks = []
chunk_size = 500
for i in range(0, len(org_list), chunk_size):
    chunk_ids = org_list[i:i+chunk_size]
    id_str = ",".join(map(str, chunk_ids))
    print(f"chunk {i} to {i+len(chunk_ids)}")
    tmp = db.raw_sql(f"""
    SELECT DISTINCT
        orgpermid,
        year,
        cusip,
        isin,
        comname
    FROM tr_esg.wrds_ref_esg
    WHERE year BETWEEN 2013 AND 2023
      AND orgpermid IN ({id_str})
    """)
    chunks.append(tmp)

chunk 0 to 500
chunk 500 to 1000
chunk 1000 to 1500
chunk 1500 to 2000
chunk 2000 to 2500
chunk 2500 to 3000
chunk 3000 to 3500
chunk 3500 to 4000
chunk 4000 to 4500
chunk 4500 to 5000
chunk 5000 to 5500
chunk 5500 to 6000
chunk 6000 to 6500
chunk 6500 to 7000
chunk 7000 to 7500
chunk 7500 to 8000
chunk 8000 to 8500
chunk 8500 to 9000
chunk 9000 to 9500
chunk 9500 to 10000
chunk 10000 to 10500
chunk 10500 to 11000
chunk 11000 to 11500
chunk 11500 to 12000
chunk 12000 to 12500
chunk 12500 to 13000
chunk 13000 to 13500
chunk 13500 to 14000
chunk 14000 to 14500
chunk 14500 to 14989


In [67]:
esg_ref = pd.concat(chunks, ignore_index=True)
esg_ref['year']   = pd.to_numeric(esg_ref['year'], errors='coerce')
esg_ref['cusip8'] = esg_ref['cusip'].astype(str).str[:8]
esg_ref = esg_ref.dropna(subset=['orgpermid', 'year', 'cusip8']).copy()
esg_ref = esg_ref.drop_duplicates(subset=['orgpermid', 'year', 'cusip8'])

In [68]:
# merge ESG 
esg_full = esg.merge(
    esg_ref,
    left_on=['orgpermid', 'fy'],
    right_on=['orgpermid', 'year'],
    how='left'
)
esg_full = esg_full.dropna(subset=['cusip8']).copy()

In [69]:
print(f"ESG full rows: {esg_full.shape[0]}")

ESG full rows: 91770


## 8. Integrate ESG into the Comp+CCM Panel

In [70]:
panel = comp_ccm.merge(
    esg_full[['cusip8', 'fy', 'esg_score']].drop_duplicates(),
    left_on=['cusip8', 'fyear'],
    right_on=['cusip8', 'fy'],
    how='left'
)

## 9. Calculate the Future Return (12 months after the end of the fiscal year)

In [71]:
print("Calculate the future return (aligned to the end date of the fiscal year)...")

Calculate the future return (aligned to the end date of the fiscal year)...


In [72]:
future_ret_df = compute_post_fy_return(
    panel['permno'].values,
    panel['datadate'].values,
    crsp_ret
)

In [73]:
panel['future_annual_ret']   = future_ret_df['future_annual_ret'].values
panel['future_n_months_ret'] = future_ret_df['future_n_months_ret'].values

## 10. Construct the Future ROA and Lag Control Variables

In [74]:
panel = panel.sort_values(['gvkey', 'fyear']).copy()

In [75]:
panel['future_roa'] = panel.groupby('gvkey')['roa'].shift(-1)
panel['lag_roa']    = panel.groupby('gvkey')['roa'].shift(1)
panel['lag_size']   = panel.groupby('gvkey')['size'].shift(1)

## 11. Final Sample Filtering

In [76]:
panel_final = panel[
    panel['esg_score'].notna() &
    (panel['at'] > 0) &
    (panel['future_n_months_ret'] >= 12)   
].copy()

panel_final['esg_score'] = panel_final['esg_score'].clip(
    lower=panel_final['esg_score'].quantile(0.01),
    upper=panel_final['esg_score'].quantile(0.99)
)

In [77]:
print(f"final sample rows: {panel_final.shape[0]}")

final sample rows: 20758


In [78]:
dup_check = panel_final.duplicated(subset=['gvkey', 'fyear']).sum()
print(dup_check)

0


In [79]:
panel_final.duplicated(subset=['gvkey', 'fyear']).sum()

np.int64(0)

## 12. Select the Final Column and Save

In [80]:
# Original column selection (before Part II additions):
# final_cols = [
#     'gvkey', 'permno', 'datadate', 'fyear', 'cusip8',
#     'sic',
#     'at', 'sale', 'ni', 'ceq',
#     'roa', 'size',
#     'lag_roa', 'lag_size',
#     'esg_score',
#     'future_roa',
#     'future_annual_ret',
#     'future_n_months_ret'
# ]

# Extended for Part II: dltt, dlc, xrd, ppent added
final_cols = [
    'gvkey', 'permno', 'datadate', 'fyear', 'cusip8',
    'sic',
    'at', 'sale', 'ni', 'ceq',
    'dltt', 'dlc', 'xrd', 'ppent',
    'roa', 'size',
    'lag_roa', 'lag_size',
    'esg_score',
    'future_roa',
    'future_annual_ret',
    'future_n_months_ret'
]

In [81]:
final_panel = panel_final[final_cols].copy()
final_panel.to_csv('esg_financial_panel_2013_2023.csv', index=False)

In [82]:
print("\n=== Descriptive statistics ===")
print(final_panel[['esg_score', 'roa', 'future_roa', 'future_annual_ret', 'at', 'size']].describe())


=== Descriptive statistics ===
       esg_score       roa  future_roa  future_annual_ret             at  \
count    20758.0   20756.0     18372.0       20758.000000        20758.0   
mean    0.404152 -0.010199   -0.006745           0.156078   21082.138004   
std     0.190924  0.227879    0.229025           0.970141  125632.181402   
min      0.06956 -9.640573   -9.640573          -0.986590           1.22   
25%     0.253776 -0.005647   -0.002773          -0.157976        727.287   
50%      0.37349  0.024326    0.025741           0.069902       2454.419   
75%     0.538747  0.069912    0.071167           0.321916     8552.91725   
max     0.854176  4.023217     1.48407          83.635370      3875393.0   

            size  
count    20758.0  
mean    7.870471  
std     1.908153  
min     0.198851  
25%     6.589321  
50%     7.805645  
75%     9.054028  
max    15.170158  


In [83]:
print("\n=== Sample size (by year)===")
print(final_panel.groupby('fyear')['gvkey'].count())


=== Sample size (by year)===
fyear
2013     745
2014     730
2015    1186
2016    1763
2017    2163
2018    2241
2019    2373
2020    2492
2021    2451
2022    2430
2023    2184
Name: gvkey, dtype: int64


In [84]:
print("\n=== ESG average (Annual)===")
print(final_panel.groupby('fyear')['esg_score'].mean())


=== ESG average (Annual)===
fyear
2013    0.437521
2014    0.443964
2015    0.396022
2016    0.369495
2017     0.36241
2018    0.367809
2019    0.385278
2020    0.404702
2021    0.428501
2022    0.442757
2023    0.440088
Name: esg_score, dtype: Float64


In [ ]:
# See how many Cusip8s in esg_full can be found in comp
esg_us = esg_full[esg_full['cusip8'].isin(comp['cusip8'])]
print(f"ESG companies that could be matched with Compustat in 2013: {esg_us[esg_us['fy']==2013].shape[0]}")

esg_ref_2013 = esg_ref[esg_ref['year'] == 2013]
us_firms = esg_ref_2013[esg_ref_2013['isin'].astype(str).str.startswith('US')]
print(f"The number of American companies in 2013: {us_firms.shape[0]}")
print(f"The total number of companies in 2013: {esg_ref_2013.shape[0]}")